<a href="https://colab.research.google.com/github/Mariam-Elbishbeashy/HeadlineGeneration-NLP/blob/main/gpt2_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets accelerate evaluate rouge_score sentencepiece -q

In [ ]:
import pandas as pd
import numpy as np
import torch

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    pipeline
)

from rouge_score import rouge_scorer

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
train_path = "/content/drive/MyDrive/headlineGenerationProjectNLP/news_headline_model_data/gpt2_train.csv"

val_path = "/content/drive/MyDrive/headlineGenerationProjectNLP/news_headline_model_data/gpt2_validation.csv"

test_path = "/content/drive/MyDrive/headlineGenerationProjectNLP/news_headline_model_data/gpt2_test.csv"

train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(42373, 2)
(5297, 2)
(5297, 2)


In [ ]:
train_df['model_input'] = train_df['model_input'].astype(str).str[:300]

val_df['model_input'] = val_df['model_input'].astype(str).str[:300]

test_df['model_input'] = test_df['model_input'].astype(str).str[:300]

In [ ]:
def format_text(row):

    return row['model_input'] + row['model_target']

train_texts = train_df.apply(format_text, axis=1)

val_texts = val_df.apply(format_text, axis=1)

test_texts = test_df.apply(format_text, axis=1)

print(train_texts.iloc[0])

Article: A unit of Chinese ride-hailing firm Didi Chuxing has submitted an application to raise 10 billion yuan ($1.6 billion) through an issuance of asset-backed securities. Didi, which said in December it had raised $4 billion to support its overseas expansion, did not respond to a Reuters' requesChina's Didi looks to raise $1.6 billion via asset-backed securities


In [ ]:
model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name
)

model.config.pad_token_id = tokenizer.pad_token_id

device='cuda' if torch.cuda.is_available() else 'cpu'

model.to(device)

# Enable T4 Tensor Core optimizations
torch.backends.cuda.matmul.allow_tf32=True


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
train_dataset = Dataset.from_dict({
    "text": train_texts.tolist()
})

val_dataset = Dataset.from_dict({
    "text": val_texts.tolist()
})

test_dataset = Dataset.from_dict({
    "text": test_texts.tolist()
})

In [ ]:
MAX_LENGTH = 48
def tokenize(batch):

    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

train_dataset = train_dataset.map(
    tokenize,
    batched=True,
    remove_columns=["text"]
)

val_dataset = val_dataset.map(
    tokenize,
    batched=True,
    remove_columns=["text"]
)

test_dataset = test_dataset.map(
    tokenize,
    batched=True,
    remove_columns=["text"]
)

Map:   0%|          | 0/42373 [00:00<?, ? examples/s]

Map:   0%|          | 0/5297 [00:00<?, ? examples/s]

Map:   0%|          | 0/5297 [00:00<?, ? examples/s]

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
    pad_to_multiple_of=8
)

In [ ]:
training_args = TrainingArguments(

    output_dir="./headline_model",

    num_train_epochs=1,

    per_device_train_batch_size=32,

    per_device_eval_batch_size=32,

    learning_rate=5e-5,

    weight_decay=0.01,

    dataloader_num_workers=2,

    logging_steps=50,

    save_steps=100000,

    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    data_collator=data_collator
)

In [ ]:
trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
50,3.181542
100,3.129751
150,3.105020
200,3.084057
250,3.069575
300,3.043392
350,3.039419
400,3.058349
450,3.030923
500,3.007157


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1325, training_loss=3.016716060998305, metrics={'train_runtime': 643.3034, 'train_samples_per_second': 65.868, 'train_steps_per_second': 2.06, 'total_flos': 1037974431744000.0, 'train_loss': 3.016716060998305, 'epoch': 1.0})

In [ ]:
torch.cuda.empty_cache()

print(
    "Allocated:",
    round(
        torch.cuda.memory_allocated()/1024**3,
        2
    ),
    "GB"
)

Allocated: 1.42 GB


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Current device:", torch.cuda.current_device())

print("Model device:", next(model.parameters()).device)

CUDA available: True
GPU: Tesla T4
Current device: 0
Model device: cuda:0


In [ ]:
save_path = "/content/drive/MyDrive/headlineGenerationProjectNLP/gpt2_model"

trainer.save_model(save_path)

tokenizer.save_pretrained(save_path)

print("Model Saved Successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model Saved Successfully!


In [ ]:
generator = pipeline(

    "text-generation",

    model=save_path,

    tokenizer=save_path,

    device=0
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [ ]:
def generate_headline(article):

    article = str(article)[:500]

    prompt = f"Article: {article} Headline:"

    result = generator(

        prompt,

        max_new_tokens=12,

        num_beams=2,

        do_sample=False
    )

    text=result[0]["generated_text"]

    headline=text.split(
        "Headline:"
    )[-1].strip()

    return headline

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import evaluate

from transformers import GPT2LMHeadModel, GPT2Tokenizer


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)


MODEL_PATH = "/content/drive/MyDrive/headlineGenerationProjectNLP/gpt2_model"

tokenizer = GPT2Tokenizer.from_pretrained(MODEL_PATH)

model = GPT2LMHeadModel.from_pretrained(MODEL_PATH)

model.to(device)

model.eval()

tokenizer.pad_token = tokenizer.eos_token

print("✅ GPT-2 model loaded successfully")


DATA_DIR = "/content/drive/MyDrive/headlineGenerationProjectNLP/news_headline_model_data"

test_df = pd.read_csv(
    os.path.join(DATA_DIR, "gpt2_test.csv")
)[["model_input", "model_target"]].dropna()

print("Test Samples:", len(test_df))


rouge = evaluate.load("rouge")


sample_size = min(200, len(test_df))

test_samples = test_df.sample(
    sample_size,
    random_state=42
).reset_index(drop=True)

inputs = test_samples["model_input"].tolist()

targets = test_samples["model_target"].tolist()


predictions = []

batch_size = 8

for i in range(0, len(inputs), batch_size):

    batch_texts = inputs[i:i + batch_size]

    prompts = [
        f"Article: {text}\nHeadline:"
        for text in batch_texts
    ]

    encodings = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256
    ).to(device)

    with torch.no_grad():

        outputs = model.generate(
            input_ids=encodings["input_ids"],
            attention_mask=encodings["attention_mask"],
            max_new_tokens=20,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=2,
            pad_token_id=tokenizer.eos_token_id
        )

    decoded_outputs = tokenizer.batch_decode(
        outputs,
        skip_special_tokens=True
    )

    for output in decoded_outputs:

        if "Headline:" in output:
            headline = output.split("Headline:")[-1].strip()
        else:
            headline = output.strip()

        headline = headline.split("\n")[0].strip()

        predictions.append(headline)


predictions = [
    p.strip()
    for p in predictions
]

targets = [
    t.strip()
    for t in targets
]

rouge_results = rouge.compute(
    predictions=predictions,
    references=targets
)

print("\n===================================")
print("ROUGE RESULTS")
print("===================================\n")

for key, value in rouge_results.items():
    print(f"{key}: {value:.4f}")


exact_matches = [
    pred.lower() == target.lower()
    for pred, target in zip(predictions, targets)
]

exact_match_rate = np.mean(exact_matches)

print("\n===================================")
print(f"Exact Match Rate: {exact_match_rate:.4f}")
print("===================================\n")


results_df = pd.DataFrame({
    "Article": inputs,
    "True Headline": targets,
    "Generated Headline": predictions,
    "Exact Match": exact_matches
})

results_path = "/content/drive/MyDrive/headlineGenerationProjectNLP/gpt2_evaluation_results.csv"

results_df.to_csv(results_path, index=False)

print(f"✅ Results saved to:\n{results_path}")


print("\n===================================")
print("SAMPLE PREDICTIONS")
print("===================================\n")

for i in range(min(5, len(results_df))):

    print("ARTICLE:")
    print(results_df.loc[i, "Article"][:300])

    print("\nTRUE HEADLINE:")
    print(results_df.loc[i, "True Headline"])

    print("\nGENERATED HEADLINE:")
    print(results_df.loc[i, "Generated Headline"])

    print("\nEXACT MATCH:")
    print(results_df.loc[i, "Exact Match"])

    print("\n" + "=" * 70 + "\n")

Using device: cuda


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

✅ GPT-2 model loaded successfully
Test Samples: 5297


A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='le


ROUGE RESULTS

rouge1: 0.4231
rouge2: 0.3581
rougeL: 0.3811
rougeLsum: 0.3811

Exact Match Rate: 0.0120

✅ Results saved to:
/content/drive/MyDrive/headlineGenerationProjectNLP/gpt2_evaluation_results.csv

SAMPLE PREDICTIONS

ARTICLE:
Article: Humans are not the only ones seeking shelter from the strong winds, heavy rains and deadly floods of Hurricane Irma. Animals, too, are finding ways to stay safe - and often, they need human help. The National Wildlife Federation says that some animals know how to take advantage of a hurrica

TRUE HEADLINE:
Dolphins, Flamingos and Pigs: The Animals Rescued From Hurricane Irma

GENERATED HEADLINE:
Article: Article: Humans are not the only ones seeking shelter from the strong winds, heavy rains and deadly floods of Hurricane Irma. Animals, too, are finding ways to stay safe - and often, they need human help. The National Wildlife Federation says that some animals know how to take advantage of a hurricane's aftermath: Raccoons scavenge for food in t